# Importing Libraries

In [1]:
import pandas as pd 
import numpy as np 
from tqdm import tqdm
import spacy
import warnings
warnings.filterwarnings('ignore')

# Retreving data

In [2]:
movies_df = pd.read_csv('../datasets/final_movies_df.csv')
tv_df = pd.read_csv('../datasets/final_tvShows_df.csv')

In [3]:
movies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11043 entries, 0 to 11042
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             11043 non-null  int64  
 1   title          11043 non-null  object 
 2   popularity     11043 non-null  float64
 3   imdb_id        11043 non-null  object 
 4   averageRating  11043 non-null  float64
 5   num_votes      11043 non-null  int64  
 6   title_type     11043 non-null  object 
 7   start_year     11043 non-null  int64  
 8   tags           11043 non-null  object 
dtypes: float64(2), int64(3), object(4)
memory usage: 776.6+ KB


In [4]:
tv_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6283 entries, 0 to 6282
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Unnamed: 0     6283 non-null   int64  
 1   id             6283 non-null   int64  
 2   title          6283 non-null   object 
 3   popularity     6283 non-null   float64
 4   imdb_id        6283 non-null   object 
 5   averageRating  6283 non-null   float64
 6   num_votes      6283 non-null   int64  
 7   title_type     6283 non-null   object 
 8   start_year     6283 non-null   int64  
 9   tags           6283 non-null   object 
dtypes: float64(2), int64(4), object(4)
memory usage: 491.0+ KB


In [5]:
df = pd.concat([movies_df,tv_df]).drop('Unnamed: 0',axis = 1)

# Building Model

## Import SpaCy NLP Model

In [6]:
nlp = spacy.load('en_core_web_lg')

In [7]:
tags = df.tags.replace(np.nan,'')

In [8]:
nlp_tags = []
for tag in tqdm(tags):
    nlp_tag = nlp(tag.strip())
    nlp_tags.append(nlp_tag)

100%|███████████████████████████████████████████████████████████████████████████| 17326/17326 [02:46<00:00, 104.31it/s]


In [9]:
df['nlp_tags'] = nlp_tags

## Searching For Movies

In [10]:
def search_movie(word):
    results = []
    for i,movie_title in enumerate(df['title']):
        if movie_title.__contains__(word):
            results.append(i)
    return df.iloc[results]

In [24]:
search_movie('Super')

,id,title,popularity,imdb_id,averageRating,num_votes,title_type,start_year,tags,nlp_tags
853,1452,Superman Returns,24.838,tt0348150,6.1,292926,movie,2006,superman collection action adventure sci-fi sa...,"(superman, collection, action, adventure, sci,..."
962,886396,Batman and Superman: Battle of the Super Sons,31.390,tt21197740,6.8,5579,video,2022,action animation sci-fi superhero villain bas...,"(action, animation, sci, -, fi, superhero, vil..."
964,34433,Dragon Ball Z: Broly – The Legendary Super Saiyan,52.991,tt0142242,7.4,12193,movie,1993,dragon ball z collection fantasy action animat...,"(dragon, ball, z, collection, fantasy, action,..."
1119,1924,Superman,39.877,tt0078346,7.4,190081,movie,1978,superman collection action adventure sci-fi ga...,"(superman, collection, action, adventure, sci,..."
1284,166076,Superman: Unbound,16.389,tt2617456,6.5,13891,video,2013,action animation adventure saving-the-world s...,"(action, animation, adventure, saving, -, the,..."
...,...,...,...,...,...,...,...,...,...,...
4614,235717,Superstar,29.582,tt29194117,6.5,10,tvSeries,2023,reality talent-show superstar reality-show for...,"(reality, talent, -, show, superstar, reality,..."
5008,90785,Superstar Singer,37.344,tt10787082,6.2,41,tvSeries,2019,reality singer finding singing ka join journe...,"(reality, , singer, finding, singing, ka, joi..."
5358,87491,DC Super Hero Girls,26.512,tt9628244,7.3,1790,tvSeries,2019,animation family sci-fi-&-fantasy kids comedy ...,"(animation, family, sci, -, fi-&-fantasy, kids..."
5732,21781,The Super Hero Squad Show,25.284,tt1388589,6.1,2114,tvSeries,2009,action-&-adventure animation comedy superhero ...,"(action-&-adventure, animation, comedy, superh..."


# Testing Recommendation

In [18]:
def recommendations_for(_id, media_type: str, year: int = 2000, least_rate: int = 5, least_num_votes: int = 1000, freq: int = 40):
    mv_df = df.query(f'start_year >= {year} & averageRating >= {least_rate} & num_votes >= {least_num_votes}').reset_index().drop(columns = 'index')
    if media_type.lower() in ['movie','tvseries']:
        target = mv_df.query(f"title_type == '{media_type}'")
    elif media_type == 'all':
        target = mv_df
    else:
        raise "You must add type (`movie`,`tvSeries`) correct or add `all`"
    movie =  mv_df.query(f"id == {_id}")
    title = movie.title.values[0]
    year = movie.start_year.values[0]
    scores = []
    indices = range(target.shape[0])
    
    print(f"Best recommendation for `{title} {int(year)}` is:")
    movie_tags = movie['nlp_tags'].values[0]
    
    for index in indices:
        scores.append(np.round(np.ceil(movie_tags.similarity(target.iloc[index].nlp_tags) * 1000) / 1000,2))
        
    recommends = pd.DataFrame({ 'id':target.id,
                                'imdb_id':target.imdb_id.values,
                                'title':target.title.values,
                                'type':target.title_type,
                                'start_year':target.start_year.values,
                                'content_score':scores,
                                'score': (scores),# + (mv_df.movie_imdb_rating / 3e1) + (mv_df.num_votes / 3.6e7) + (mv_df.start_year / 5e2),
                                'rating':target.averageRating,
                                'popularity':target.popularity,
                                'num_votes':target.num_votes,
                                'tags':target.tags
                              })
    
    recommends.drop(recommends.query('content_score >= 1.0').index,inplace = True)
    recommends = recommends.sort_values(by=['score'],ascending = False).reset_index().drop(columns = ['index','score'])
    return recommends.head(freq)

In [23]:
recommendations_for(1396,'movie')

Best recommendation for `Breaking Bad 2008` is:


,id,imdb_id,title,type,start_year,content_score,rating,popularity,num_votes,tags
0,39013,tt1399683,Winter's Bone,movie,2010,0.96,7.1,12.577,151794,mystery crime drama sheriff based-on-novel-or...
1,641,tt0180093,Requiem for a Dream,movie,2000,0.96,8.3,17.020,912336,crime drama drug-dealer corruption drug-abuse...
2,833425,tt7550014,No Exit,movie,2022,0.95,6.1,23.013,33962,mystery thriller drama based-on-novel-or-book...
3,41446,tt1262416,Scream 4,movie,2011,0.95,6.2,37.485,173793,scream collection mystery horror rescue mask s...
4,271718,tt3152624,Trainwreck,movie,2015,0.95,6.2,22.510,143738,comedy romance drama alcohol surgeon intervie...
5,59965,tt1600195,Abduction,movie,2011,0.95,5.1,14.567,83845,mystery action thriller drama central-intelli...
6,12797,tt0995039,Ghost Town,movie,2008,0.95,6.7,14.408,78292,fantasy comedy drama dying-and-death adultery...
7,800158,tt1136617,The Killer,movie,2023,0.95,6.7,26.335,195758,crime action adventure thriller sniper new-yo...
8,257091,tt2561572,Get Hard,movie,2015,0.95,6.0,19.412,148560,crime comedy action drama prison fbi training...
9,164457,tt1206543,Out of the Furnace,movie,2013,0.95,6.7,11.530,126406,crime action thriller drama prison drug-deale...


# Storing Result into Cloud DataBase